# Stage 4: Train YOLO26-Pico-P2 with Pretrained Stem & Hard Negatives (`04_train_pretrained_with_negatives.ipynb`)

### Experiment 4: `exp4_pretrained_negatives` (Model Registry `m004`)

This notebook trains our high-resolution edge detector using both **COCO pretrained stem/backbone transfer** and **automated hard negative background mining** (~25% informative vehicle-free tiles: roofs, crosswalks, shadows, highway dividers).

Target Outcomes:
- Drastic reduction in False Alarms / min across aerial scenes.
- Higher precision on small, distant vehicles at `200–400 m`.
- Complete preservation of Rule 1 (test split isolation) and Rule 4 (dual distance bands).

In [ ]:
# === CELL 1: GPU & REPOSITORY SETUP ===
import os
import sys
from pathlib import Path
import torch

GITHUB_REPO_URL = "https://github.com/ituvtu/aerial-vehicle-detector.git"
REPO_DIR = Path("/content/aerial-vehicle-detector")

if not REPO_DIR.exists():
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only origin main

%cd {REPO_DIR}

!pip install -q "ultralytics>=8.4.0" "git+https://github.com/ultralytics/CLIP.git" "pandas" "pyyaml" "matplotlib"

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    !nvidia-smi

In [ ]:
# === CELL 2: DATASET SYNCHRONIZATION ===
import yaml
from src.manifest_manager import ensure_images_available, ensure_raw_eval_video_available, sync_manifest_splits

with open("configs/project_config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

ensure_images_available(
    manifest_path=cfg["paths"]["manifest_path"],
    images_dir=cfg["paths"]["images_dir"],
    bundle_zip_path=cfg["paths"]["images_bundle_zip"],
    bundle_url=cfg.get("remote_data", {}).get("images_bundle_url"),
)
split_counts = sync_manifest_splits()
print("Frames unpacked. Split distribution:", split_counts)

# Ensure raw 30 FPS evaluation video is ready for demo generation
raw_eval_video = ensure_raw_eval_video_available("32179597")
print(f"Raw evaluation video ready: {raw_eval_video} ({raw_eval_video.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# === CELL 3: AUTOMATED HARD NEGATIVE BACKGROUND MINING ===
# Extracts 24 diverse, high-texture 1024x1024 tiles with strictly zero GT vehicle overlap
!python scripts/03b_generate_hard_negatives.py --max-negatives 24

import cv2
import matplotlib.pyplot as plt
from pathlib import Path

neg_samples = list(Path("data/images_negatives").glob("*.jpg"))[:4]
if neg_samples:
    fig, axes = plt.subplots(1, len(neg_samples), figsize=(18, 5))
    for ax, p in zip(axes, neg_samples):
        img = cv2.imread(str(p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(p.name, fontsize=9)
        ax.axis("off")
    plt.suptitle("Sample Extracted Hard Negative Tiles (Zero GT Overlap)", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# === CELL 4: ARCHITECTURE & PRETRAINED STEM TRANSFER VERIFICATION ===
from ultralytics import YOLO
from src.metrics import inspect_and_verify_architecture, transfer_coco_stem_weights

# Audit architecture (<500k params, Rule 5)
arch_report = inspect_and_verify_architecture(
    yaml_path="configs/yolo26_pico_p2.yaml",
    pretrained_weights="yolo26n.pt",
    output_report_path="results/model_architecture_report.json",
    max_params=500_000,
)

print(f"\nArchitecture Verified: {arch_report['unfused_params']:,} unfused / {arch_report['fused_params']:,} fused params")

# Verify deterministic PyTorch stem transfer
test_model = YOLO("configs/yolo26_pico_p2.yaml")
transfer_info = transfer_coco_stem_weights(test_model, weights_path="yolo26n.pt")
print("Transfer Verification:", transfer_info)

In [ ]:
# === CELL 5: TRAINING WITH PRETRAINED STEM & HARD NEGATIVES (exp4) ===
# Trains on 73 vehicle frames + 24 background hard negative tiles (97 total train images)
!python scripts/04_train.py --mode exp4_pretrained_negatives --pretrained-stem --data data/dataset_with_negatives.yaml --epochs 150 --batch 8 --imgsz 1024

In [ ]:
# === CELL 6: TRAINING TELEMETRY & MODEL REGISTRY REVIEW ===
from IPython.display import Image, display
from pathlib import Path

!python scripts/04_train.py --list-models

curves_p = Path("results/training_curves_exp4_pretrained_negatives.png")
if curves_p.exists():
    display(Image(filename=str(curves_p)))

In [ ]:
# === CELL 7: INDEPENDENT TEST EVALUATION (m004) ===
# Decoupled evaluation on held-out test split (eval + eval_B)
!python scripts/05_evaluate.py --weights weights/exp4_pretrained_negatives_best.pt --calibration weights/calibration_val_exp4_pretrained_negatives.json

from IPython.display import Markdown, display
from pathlib import Path

eval_summary_p = Path("results/eval_summary_table.md")
if eval_summary_p.exists():
    display(Markdown(eval_summary_p.read_text(encoding="utf-8")))

In [ ]:
# === CELL 8: FULL 30 FPS FULL-HD EVALUATION DEMO VIDEO ===
!python scripts/05_evaluate.py --weights weights/exp4_pretrained_negatives_best.pt --calibration weights/calibration_val_exp4_pretrained_negatives.json

demo_video_p = Path("results/eval_demo_32179597_full_fps.mp4")
if demo_video_p.exists():
    print(f"\nFull 30.0 FPS Evaluation Video generated: {demo_video_p} ({demo_video_p.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# === CELL 9: PACKAGE & DOWNLOAD FINAL DELIVERABLES BUNDLE ===
import datetime
import zipfile
from pathlib import Path

timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
bundle_zip = Path(f"colab_results_bundle_m004_champion_{timestamp}.zip")

include_dirs = [Path("weights"), Path("results"), Path("assets"), Path("data/registry")]

with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in include_dirs:
        if folder.exists():
            for item in folder.rglob("*"):
                if item.is_file() and not item.name.endswith(".mp4") and item.stat().st_size < 50_000_000:
                    zf.write(item, arcname=item.as_posix())

print(f"Packaged deliverables bundle: {bundle_zip} ({bundle_zip.stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files
    files.download(str(bundle_zip))
    print("Download triggered in browser.")
except (ImportError, Exception) as exc:
    print(f"Note: Running locally or download skipped: {exc}. Bundle saved at {bundle_zip}")